In [1]:
##Install
%pip install opendatasets --upgrade --quiet
%pip install legacy-cgi
%pip install scikit-learn
%pip install pandas

#XGBoost
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import opendatasets as od
UCI_Poker = 'https://www.kaggle.com/datasets/rasvob/uci-poker-hand-dataset/data?select=poker-hand-testing.data'
od.download(UCI_Poker)

Skipping, found downloaded files in ".\uci-poker-hand-dataset" (use force=True to force download)


In [3]:
import os
import pandas as pd

# Ensure dataset downloaded
os.listdir('./uci-poker-hand-dataset')


['poker-hand-testing.data', 'poker-hand-training-true.data']

In [4]:
##Process Data
import xgboost as xgb
from sklearn.model_selection import GridSearchCV 

data_train = pd.read_csv(filepath_or_buffer="./uci-poker-hand-dataset/poker-hand-training-true.data", sep=',', header=None)
data_test = pd.read_csv(filepath_or_buffer="./uci-poker-hand-dataset/poker-hand-testing.data", sep=',', header=None)

X_train = data_train.values[:,0:10] #all rows of first 10 cols
y_train = data_train.values[:,10] #all rows of 11th col

X_test = data_test.values[:,0:10]
y_test = data_test.values[:,10]


##Tune model
#Gamma and subsample left out to speedup tuning time since their best values consistently result in their respective defaults
params = {
    'max_depth': [3, 6], #Over 6 is more likely to overfit it seems
    'eta': [0.1, 0.2, 0.3, 0.4, 0.5],
    'min_child_weight': [1, 2],
    # 'gamma': [0, 1, 3],
    # 'subsample': [0.5, 0.7, 1]
}

model = xgb.XGBClassifier(objective='multi:softprob', num_class=10)

print("Starting tuning")
grid = GridSearchCV(model, params, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Best set of hyperparameters: ", grid.best_params_)
print("Best score: ", grid.best_score_)

Starting tuning
Best set of hyperparameters:  {'eta': 0.5, 'max_depth': 6, 'min_child_weight': 1}
Best score:  0.7849260295881647


In [5]:
##Metrics
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

train_preds = grid.predict(X_train)
test_preds = grid.predict(X_test)

accuracy_train = accuracy_score(y_train, train_preds)
accuracy_test = accuracy_score(y_test, test_preds)
print(f"Train Accuracy: {accuracy_train:.2f}")
print(f"Test Accuracy: {accuracy_test:.2f}")
print()

f1_train = f1_score(y_train, train_preds, average="weighted")
f1_test = f1_score(y_test, test_preds, average="weighted")
print(f"F1 Score (train): {f1_train:.2f}")
print(f"F1 Score (test): {f1_test:.2f}")

# Confusion matrix and classification report
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test, test_preds))
print("\nClassification Report (Test):")
print(classification_report(y_test, test_preds, digits=3, zero_division=0))

Train Accuracy: 0.94
Test Accuracy: 0.80

F1 Score (train): 0.94
F1 Score (test): 0.77
Confusion Matrix (Test):
[[468329  32732     90     20     27     10      1      0      0      0]
 [ 92575 322046   6606    861    327     20     16     25      6     16]
 [    99  43323   3654    474     61      0      8      3      0      0]
 [     7  18563    833   1663     37      0     11      4      1      2]
 [   630   3071     69      8     95      0      1      0      1     10]
 [  1644     45      0      0      0    307      0      0      0      0]
 [     0   1012    252    153      2      0      5      0      0      0]
 [     0    123     19     87      0      0      1      0      0      0]
 [     2      8      0      0      0      2      0      0      0      0]
 [     1      2      0      0      0      0      0      0      0      0]]

Classification Report (Test):
              precision    recall  f1-score   support

           0      0.831     0.934     0.880    501209
           1     

# Metrics Output:
Train Accuracy: 0.94

Test Accuracy: 0.80

F1 Score (train): 0.94

F1 Score (test): 0.77


Definitely overfit but we still get 80% accuracy and a solid F1 score. This F1 score is a good sign since the dataset is very imbalanced. Reducing our max_depth below 6 definitely clamps the overfitting but we get overall far worse accuracies. Gamma can also be increased to substantially reduce the overfitting but once again that drops our accuracy down to around 50%.